In [48]:
import pandas as pd
import numpy as np


In [50]:
df=pd.read_csv("DateFruit_Dataset.csv")


In [52]:
df.head() #34 feature ,#1 category

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [54]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [56]:
df.shape

(898, 35)

In [58]:
X=df.drop("Class",axis=1)
y=df["Class"]

In [60]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [62]:
from sklearn.preprocessing import StandardScaler , LabelEncoder

le = LabelEncoder()
y=le.fit_transform(y)

In [64]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test =train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [66]:
scaler=StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [68]:
X_train_scaled.shape

(718, 34)

# ANN classifier

In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset , DataLoader

In [ ]:
#firstly we convert the datsets in tensor

X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor =torch.tensor(y_train,dtype=torch.long)

X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [ ]:
#implemented the dataloader and tensordataloder
#tensor dataset loding
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

#Dataloader
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [ ]:
#build the model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        
        self.model= nn.Sequential(
            #1st layer
           nn.Linear(X.shape[1],64),
           nn.ReLU(),
            #2nd layer
           nn.Linear(64,64),
           nn.ReLU(),
            #3rd layer
           nn.Linear(64,7)
        
          )
    def forward(self,x):
        return self.model(x)

In [ ]:
model=ANN()

#loss & optimzer
criteria = nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [ ]:
#training the NN

epochs=100

for epoch in range(epochs):
    model.train()

    running_loss=0.0

    for xb,yb in train_loader:
        
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss=criteria(outputs,yb) 
        loss.backward() 
        optimizer.step() #parameters updation steps
        running_loss += loss.item()

    train_loss=running_loss/len(train_loader)

    print(f"epoch={epoch}/{epochs}, loss={train_loss}")
    

In [ ]:
#evaluation
model.eval()
total=0
correct=0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb)
        _,predicated = torch.max(outputs,1)

        correct +=(predicated == yb).sum().item()
        total+=yb.size(0) #actual samples in each batch

print("total_values: ",total)    
print("corrected_values: ",correct)    

In [ ]:
print("accuracy: ",correct/total*100)

# PCA dimensionality reducation to imporve the model perforamnce


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA

# 1. Define the component values you want to test
components_to_test = [21, 22, 23, 24, 25,26,27,28,29,30,31,32,34,33]
results = {}

# Define your ANN structure dynamically based on input size
class ANN(nn.Module):
    def __init__(self, input_dim):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7) # 7 Output classes
        )
    def forward(self, x):
        return self.model(x)

# 2. Start the loop
for n in components_to_test:
    print(f"\n--- Training ANN with {n} PCA Components ---")
    
    # Apply PCA for this specific 'n'
    pca = PCA(n_components=n)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    # Convert to PyTorch Tensors
    X_train_tensor = torch.tensor(X_train_pca, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values if hasattr(y_train, 'values') else y_train, dtype=torch.long)
    X_test_tensor  = torch.tensor(X_test_pca, dtype=torch.float32)
    y_test_tensor  = torch.tensor(y_test.values if hasattr(y_test, 'values') else y_test, dtype=torch.long)
    
    # Prepare DataLoaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    # Initialize the model with the current number of components (n)
    model = ANN(input_dim=n)
    criteria = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Train the model (Keep epochs lower inside a loop to save time, e.g., 30-50 epochs)
    epochs = 50 
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criteria(outputs, yb)
            loss.backward()
            optimizer.step()
            
    # Evaluate accuracy on the test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            outputs = model(xb)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == yb).sum().item()
            total += yb.size(0)
            
    accuracy = (correct / total) * 100
    results[n] = accuracy
    print(f"✅ Components: {n} -> Test Accuracy: {accuracy:.2f}%")

# 3. Print final comparison summary
print("\n====== FINAL ACCURACY SUMMARY ======")
for comp, acc in results.items():
    print(f"PCA Components: {comp} | Accuracy: {acc:.2f}%")
